In [1]:
import pandas as pd
from pathlib import Path

# Load and combine the 6 part CSVs (same schema)
parts = sorted(Path("new_guardian_articles").glob("articledb_part*.csv"))
if not parts:
    raise FileNotFoundError("No files found in new_guardian_articles/articledb_part*.csv")

df = pd.concat((pd.read_csv(p) for p in parts), ignore_index=True)

# Parse date
df["Publication Date"] = pd.to_datetime(df["Publication Date"], utc=True, errors="coerce")

# Example: frequency of Politics articles by year
politics = df[df["Article Category"].astype(str).str.strip().eq("Politics")].copy()
politics["year"] = politics["Publication Date"].dt.year
print(politics["year"].value_counts().sort_index())


year
1984.0      1
1994.0      1
1995.0      1
1997.0      2
1999.0      6
2000.0     15
2001.0     11
2002.0      5
2003.0      4
2004.0      8
2005.0     18
2006.0      4
2007.0     14
2008.0     40
2009.0     46
2010.0     57
2011.0     31
2012.0     76
2013.0     92
2014.0    175
2015.0    191
2016.0    386
2017.0    431
2018.0    218
2019.0    449
2020.0    178
2021.0    218
2022.0    378
2023.0    704
2024.0    623
Name: count, dtype: int64


In [2]:
import pandas as pd
from pathlib import Path

# Existing extended dataset (your master file)
old_path = Path("data/guardian_articles_extended.csv")
old_df = pd.read_csv(old_path)

# Parse date
old_df["Publication Date"] = pd.to_datetime(old_df["Publication Date"], utc=True, errors="coerce")

print("Existing dataset rows:", len(old_df))


Existing dataset rows: 138236


/var/folders/gr/j8nwj3f15hgg45p8gkw2c5b00000gn/T/ipykernel_38496/2278531249.py:6: DtypeWarning: Columns (0,1,2,3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  old_df = pd.read_csv(old_path)


In [3]:
# Min / max date ranges (existing vs new)
old_dates = old_df["Publication Date"].dropna()
new_dates = df["Publication Date"].dropna()

print("Existing date range:", old_dates.min(), "->", old_dates.max())
print("New date range:", new_dates.min(), "->", new_dates.max())


Existing date range: 1855-06-30 02:47:00+00:00 -> 2024-05-07 21:38:32+00:00
New date range: 1855-06-30 02:47:00+00:00 -> 2024-05-07 21:38:32+00:00


In [4]:
import pandas as pd
from pathlib import Path

out_path = Path("data/guardian_articles_extended_1990_2024_dedup.csv")

EXPECTED_COLS = [
    "URL","Article Category","Publication Date","Article Author",
    "Article Title","Article Contents","Data Quality"
]

def standardize_guardian_df(x: pd.DataFrame) -> pd.DataFrame:
    """Standardize either schema into the canonical 7-column schema."""
    x = x.copy()

    # If this is the API-style schema, map it
    if "webUrl" in x.columns or "webPublicationDate" in x.columns:
        x = x.rename(columns={
            "webUrl": "URL",
            "sectionName": "Article Category",
            "webPublicationDate": "Publication Date",
            "webTitle": "Article Title",
            "bodyContent": "Article Contents",
        })

    # Ensure required columns exist
    for col in EXPECTED_COLS:
        if col not in x.columns:
            x[col] = pd.NA

    # Keep only expected columns (consistent order)
    x = x[EXPECTED_COLS].copy()

    # Normalize URL/date
    x["URL"] = x["URL"].astype(str).str.strip()
    x["Publication Date"] = pd.to_datetime(x["Publication Date"], utc=True, errors="coerce")

    return x

# Standardize both datasets
old_std = standardize_guardian_df(old_df)
new_std = standardize_guardian_df(df)   # df is your combined part1-6 file

# Combine
combined = pd.concat([old_std, new_std], ignore_index=True)

# Filter to date range 1990-2024 (inclusive)
start = pd.Timestamp("1990-01-01", tz="UTC")
end = pd.Timestamp("2024-12-31 23:59:59", tz="UTC")
combined = combined.loc[combined["Publication Date"].between(start, end)].copy()

# Dedupe by URL, keeping the 'best' record:
# prefer higher Data Quality (if numeric), then longer content, then newer date
combined["_content_len"] = combined["Article Contents"].astype(str).str.len()
combined["_dq_num"] = pd.to_numeric(combined["Data Quality"], errors="coerce")

combined = combined.sort_values(
    by=["URL", "_dq_num", "_content_len", "Publication Date"],
    ascending=[True, False, False, False],
    kind="mergesort"
)
combined = combined.drop_duplicates(subset=["URL"], keep="first").drop(columns=["_content_len", "_dq_num"])

combined.to_csv(out_path, index=False)
print(f"Saved: {out_path} | rows={len(combined):,}")
print("Final date range:", combined["Publication Date"].min(), "->", combined["Publication Date"].max())


Saved: data/guardian_articles_extended_1990_2024_dedup.csv | rows=112,316
Final date range: 1991-01-21 02:21:50+00:00 -> 2024-05-07 21:38:32+00:00
